In [30]:
import pandas as pd

feature_df = pd.read_csv(
    "D:\\Coding\\6th sem\\Microprocessor\\road_quality\\preprocessed_data\\road_feature_dataset_50Hz_2sec_50overlap.csv"
)

In [31]:
#select only required columns
rst_df = feature_df[[
    "acc_std",
    "acc_rms",
    "z_peak_to_peak",
    "road_label"
]].copy()

In [32]:
#discretize
for col in ["acc_std", "acc_rms", "z_peak_to_peak"]:
    rst_df[col] = pd.qcut(
        rst_df[col],
        q=3,
        labels=["Low", "Medium", "High"]
    )

In [33]:
#conflict detection
groups = rst_df.groupby(
    ["acc_std", "acc_rms", "z_peak_to_peak"],
    observed=True  # fixes future warning
)

decision_table = groups["road_label"].apply(list).reset_index()

def check_conflict(x):
    if isinstance(x, list):
        return len(set(x)) > 1
    return False

decision_table["conflict"] = decision_table["road_label"].apply(check_conflict)

print(decision_table)

   acc_std acc_rms z_peak_to_peak  \
0      Low     Low            Low   
1      Low     Low         Medium   
2      Low  Medium            Low   
3      Low  Medium         Medium   
4   Medium     Low            Low   
5   Medium     Low         Medium   
6   Medium     Low           High   
7   Medium  Medium            Low   
8   Medium  Medium         Medium   
9   Medium  Medium           High   
10  Medium    High         Medium   
11  Medium    High           High   
12    High     Low           High   
13    High  Medium         Medium   
14    High  Medium           High   
15    High    High         Medium   
16    High    High           High   

                                           road_label  conflict  
0   [smooth, smooth, smooth, smooth, smooth, smoot...      True  
1   [smooth, smooth, smooth, smooth, smooth, smoot...     False  
2   [smooth, smooth, smooth, smooth, smooth, smoot...      True  
3                            [smooth, smooth, smooth]     False  
4  

In [34]:
print(rst_df.isnull().sum())

acc_std           0
acc_rms           0
z_peak_to_peak    0
road_label        0
dtype: int64


In [35]:
print("Total combinations:", len(decision_table))
print("Conflicting combinations:", decision_table["conflict"].sum())

Total combinations: 17
Conflicting combinations: 11


In [36]:
#Lower Approximation Objects Count
lower_combinations = decision_table[decision_table["conflict"] == False]

# Count total objects in lower approximation
lower_count = 0
for labels in lower_combinations["road_label"]:
    lower_count += len(labels)

print("Lower Approximation Objects:", lower_count)

Lower Approximation Objects: 22


In [37]:
#Upper Approximation Objects Count
upper_count = len(rst_df)
print("Upper Approximation Objects:", upper_count)

Upper Approximation Objects: 166


In [38]:
#Boundary Region Size
boundary_count = upper_count - lower_count
print("Boundary Region Objects:", boundary_count)

Boundary Region Objects: 144


In [39]:
#dependency degree
dependency_degree = lower_count / upper_count
print("Dependency Degree:", round(dependency_degree, 4))

Dependency Degree: 0.1325


In [40]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(max_depth=3)
tree.fit(feature_df[["acc_std"]], feature_df["road_label"])

,criterion,'gini'
,splitter,'best'
,max_depth,3
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [53]:
import numpy as np

# Remove leaf indicators
thresholds = tree.tree_.threshold
thresholds = thresholds[thresholds != -2]

# Remove duplicates and sort
thresholds = sorted(set(thresholds))

print("Unique thresholds:", thresholds)

Unique thresholds: [np.float64(1852.3029174804688), np.float64(3051.450439453125), np.float64(3060.6904296875), np.float64(3445.41162109375), np.float64(3947.046142578125), np.float64(3975.7113037109375), np.float64(4097.880615234375)]


In [54]:
# Pick 2 middle thresholds
# Pick roughly spaced thresholds
selected_thresholds = [
    thresholds[1],   # around 3051
    thresholds[4]    # around 3947
]

print("Selected thresholds:", selected_thresholds)

Selected thresholds: [np.float64(3051.450439453125), np.float64(3947.046142578125)]


In [55]:
bins = [-np.inf] + selected_thresholds + [np.inf]

labels = ["Low", "Medium", "High"]

feature_df["acc_std_bin"] = pd.cut(
    feature_df["acc_std"],
    bins=bins,
    labels=labels
)

In [56]:
#Repeat Threshold Extraction for Other Features
# ---- ACC_RMS ----
X_rms = feature_df[["acc_rms"]]
y = feature_df["road_label"]

tree_rms = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_rms.fit(X_rms, y)

thresholds_rms = tree_rms.tree_.threshold
thresholds_rms = thresholds_rms[thresholds_rms != -2]
thresholds_rms = sorted(set(thresholds_rms))

print("RMS thresholds:", thresholds_rms)

RMS thresholds: [np.float64(16162.28759765625), np.float64(16264.6337890625), np.float64(16569.8271484375), np.float64(16603.2373046875), np.float64(16752.3408203125), np.float64(16918.8330078125), np.float64(17216.71484375)]


In [57]:
selected_rms = [thresholds_rms[1], thresholds_rms[4]]

bins_rms = [-np.inf] + selected_rms + [np.inf]

feature_df["acc_rms_bin"] = pd.cut(
    feature_df["acc_rms"],
    bins=bins_rms,
    labels=["Low", "Medium", "High"]
)

In [58]:
#Build Final RST Table Using Only BIN Columns

In [60]:
#Step 1 — Train Tree for z_peak_to_peak
X_z = feature_df[["z_peak_to_peak"]]
y = feature_df["road_label"]

tree_z = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_z.fit(X_z, y)

thresholds_z = tree_z.tree_.threshold
thresholds_z = thresholds_z[thresholds_z != -2]
thresholds_z = sorted(set(thresholds_z))

print("Z thresholds:", thresholds_z)

Z thresholds: [np.float64(13336.0), np.float64(17276.0), np.float64(17382.0), np.float64(18416.0), np.float64(19108.0), np.float64(24376.0), np.float64(26908.0)]


In [61]:
#Step 2 — Select Two Good Thresholds
selected_z = [
    thresholds_z[1],
    thresholds_z[4]
]

print("Selected Z thresholds:", selected_z)

Selected Z thresholds: [np.float64(17276.0), np.float64(19108.0)]


In [63]:
bins_z = [-np.inf] + selected_z + [np.inf]

feature_df["z_peak_to_peak_bin"] = pd.cut(
    feature_df["z_peak_to_peak"],
    bins=bins_z,
    labels=["Low", "Medium", "High"]
)
print(feature_df.columns)

Index(['road_label', 'acc_mean', 'acc_std', 'acc_rms', 'z_max', 'z_min',
       'z_peak_to_peak', 'speed_mean', 'speed_std', 'acc_std_bin',
       'acc_rms_bin', 'z_peak_to_peak_bin'],
      dtype='object')


In [65]:
rst_df = feature_df[[
    "acc_std_bin",
    "acc_rms_bin",
    "z_peak_to_peak_bin",
    "road_label"
]].copy()

rst_df = rst_df.dropna()
print(rst_df)

    acc_std_bin acc_rms_bin z_peak_to_peak_bin road_label
0           Low         Low                Low     smooth
1           Low      Medium                Low     smooth
2        Medium      Medium               High     smooth
3           Low      Medium               High     smooth
4           Low        High               High     smooth
..          ...         ...                ...        ...
161      Medium        High               High      rough
162      Medium        High               High      rough
163        High      Medium             Medium      rough
164        High        High               High      rough
165        High        High               High      rough

[166 rows x 4 columns]


In [66]:
#rebuild decision Table

In [67]:
groups = rst_df.groupby(
    ["acc_std_bin", "acc_rms_bin", "z_peak_to_peak_bin"],
    observed=True
)

decision_table = groups["road_label"].apply(list).reset_index()

decision_table["conflict"] = decision_table["road_label"].apply(
    lambda x: len(set(x)) > 1
)

print(decision_table)

   acc_std_bin acc_rms_bin z_peak_to_peak_bin  \
0          Low         Low                Low   
1          Low         Low             Medium   
2          Low         Low               High   
3          Low      Medium                Low   
4          Low      Medium             Medium   
5          Low      Medium               High   
6          Low        High                Low   
7          Low        High             Medium   
8          Low        High               High   
9       Medium         Low               High   
10      Medium      Medium                Low   
11      Medium      Medium             Medium   
12      Medium      Medium               High   
13      Medium        High               High   
14        High         Low               High   
15        High      Medium             Medium   
16        High      Medium               High   
17        High        High             Medium   
18        High        High               High   

                   

In [68]:
#compute Roughset Metrics
# Lower Approximation
lower_combinations = decision_table[decision_table["conflict"] == False]

lower_count = 0
for labels in lower_combinations["road_label"]:
    lower_count += len(labels)

upper_count = len(rst_df)
boundary_count = upper_count - lower_count

dependency_degree = lower_count / upper_count

print("Lower Approximation Objects:", lower_count)
print("Boundary Region Objects:", boundary_count)
print("Dependency Degree:", round(dependency_degree, 4))

Lower Approximation Objects: 35
Boundary Region Objects: 131
Dependency Degree: 0.2108


In [69]:
X_speed = feature_df[["speed_mean"]]
y = feature_df["road_label"]

tree_speed = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_speed.fit(X_speed, y)

thresholds_speed = tree_speed.tree_.threshold
thresholds_speed = thresholds_speed[thresholds_speed != -2]
thresholds_speed = sorted(set(thresholds_speed))

print("Speed thresholds:", thresholds_speed)

Speed thresholds: [np.float64(5.837049961090088), np.float64(6.568700075149536), np.float64(14.955450057983398), np.float64(18.427949905395508), np.float64(21.15334987640381), np.float64(22.720900535583496)]


In [70]:
#select threshold
selected_speed = [
    thresholds_speed[1],   # ~6.56
    thresholds_speed[3]    # ~18.42
]

print("Selected speed thresholds:", selected_speed)

Selected speed thresholds: [np.float64(6.568700075149536), np.float64(18.427949905395508)]


In [71]:
#create speed bins
import numpy as np

bins_speed = [-np.inf] + selected_speed + [np.inf]

feature_df["speed_mean_bin"] = pd.cut(
    feature_df["speed_mean"],
    bins=bins_speed,
    labels=["Low", "Medium", "High"]
)

print(feature_df[["speed_mean", "speed_mean_bin"]].head())

   speed_mean speed_mean_bin
0      4.5741            Low
1      6.0115            Low
2      9.2814         Medium
3     13.9328         Medium
4     15.9659         Medium


In [72]:
#hybrid RST table
rst_df = feature_df[[
    "acc_std_bin",
    "acc_rms_bin",
    "z_peak_to_peak_bin",
    "speed_mean_bin",
    "road_label"
]].copy()

rst_df = rst_df.dropna()

In [73]:
#recompute decision table
groups = rst_df.groupby(
    ["acc_std_bin", "acc_rms_bin", "z_peak_to_peak_bin", "speed_mean_bin"],
    observed=True
)

decision_table = groups["road_label"].apply(list).reset_index()

decision_table["conflict"] = decision_table["road_label"].apply(
    lambda x: len(set(x)) > 1
)

In [74]:
#compute new dependencies
lower_combinations = decision_table[decision_table["conflict"] == False]

lower_count = 0
for labels in lower_combinations["road_label"]:
    lower_count += len(labels)

upper_count = len(rst_df)
boundary_count = upper_count - lower_count
dependency_degree = lower_count / upper_count

print("Lower Approximation Objects:", lower_count)
print("Boundary Region Objects:", boundary_count)
print("Dependency Degree:", round(dependency_degree, 4))

Lower Approximation Objects: 135
Boundary Region Objects: 31
Dependency Degree: 0.8133


In [75]:
#Finding Reducts

In [76]:
import itertools

def compute_dependency(df, condition_cols):
    
    groups = df.groupby(condition_cols, observed=True)
    decision_table = groups["road_label"].apply(list).reset_index()
    
    decision_table["conflict"] = decision_table["road_label"].apply(
        lambda x: len(set(x)) > 1
    )
    
    lower = decision_table[decision_table["conflict"] == False]
    
    lower_count = 0
    for labels in lower["road_label"]:
        lower_count += len(labels)
    
    dependency = lower_count / len(df)
    
    return dependency

In [77]:
#compute reduct
features = [
    "acc_std_bin",
    "acc_rms_bin",
    "z_peak_to_peak_bin",
    "speed_mean_bin"
]

full_dependency = compute_dependency(rst_df, features)

print("Full-set dependency:", round(full_dependency, 4))

best_subset = None

for r in range(1, len(features)+1):
    for subset in itertools.combinations(features, r):
        
        dep = compute_dependency(rst_df, list(subset))
        
        print("Subset:", subset, "Dependency:", round(dep,4))
        
        # Keep first minimal subset reaching full dependency
        if dep >= full_dependency:
            best_subset = subset
            break
    
    if best_subset is not None:
        break

print("\nHybrid Reduct:", best_subset)

Full-set dependency: 0.8133
Subset: ('acc_std_bin',) Dependency: 0.0
Subset: ('acc_rms_bin',) Dependency: 0.0
Subset: ('z_peak_to_peak_bin',) Dependency: 0.0
Subset: ('speed_mean_bin',) Dependency: 0.0
Subset: ('acc_std_bin', 'acc_rms_bin') Dependency: 0.0542
Subset: ('acc_std_bin', 'z_peak_to_peak_bin') Dependency: 0.1867
Subset: ('acc_std_bin', 'speed_mean_bin') Dependency: 0.3373
Subset: ('acc_rms_bin', 'z_peak_to_peak_bin') Dependency: 0.0241
Subset: ('acc_rms_bin', 'speed_mean_bin') Dependency: 0.3373
Subset: ('z_peak_to_peak_bin', 'speed_mean_bin') Dependency: 0.1084
Subset: ('acc_std_bin', 'acc_rms_bin', 'z_peak_to_peak_bin') Dependency: 0.2108
Subset: ('acc_std_bin', 'acc_rms_bin', 'speed_mean_bin') Dependency: 0.6928
Subset: ('acc_std_bin', 'z_peak_to_peak_bin', 'speed_mean_bin') Dependency: 0.5361
Subset: ('acc_rms_bin', 'z_peak_to_peak_bin', 'speed_mean_bin') Dependency: 0.4337
Subset: ('acc_std_bin', 'acc_rms_bin', 'z_peak_to_peak_bin', 'speed_mean_bin') Dependency: 0.8133


In [78]:
#RULE EXTRACTION
certain_rules = decision_table[decision_table["conflict"] == False]

for _, row in certain_rules.iterrows():
    condition = (
        f"IF acc_std is {row['acc_std_bin']} AND "
        f"acc_rms is {row['acc_rms_bin']} AND "
        f"z_peak_to_peak is {row['z_peak_to_peak_bin']} AND "
        f"speed is {row['speed_mean_bin']}"
    )
    
    decision = list(set(row["road_label"]))[0]
    
    print(condition, "THEN road =", decision)

IF acc_std is Low AND acc_rms is Low AND z_peak_to_peak is Low AND speed is High THEN road = smooth
IF acc_std is Low AND acc_rms is Low AND z_peak_to_peak is Medium AND speed is High THEN road = smooth
IF acc_std is Low AND acc_rms is Low AND z_peak_to_peak is High AND speed is Medium THEN road = smooth
IF acc_std is Low AND acc_rms is Low AND z_peak_to_peak is High AND speed is High THEN road = smooth
IF acc_std is Low AND acc_rms is Medium AND z_peak_to_peak is Low AND speed is Low THEN road = smooth
IF acc_std is Low AND acc_rms is Medium AND z_peak_to_peak is Medium AND speed is High THEN road = smooth
IF acc_std is Low AND acc_rms is Medium AND z_peak_to_peak is High AND speed is Medium THEN road = smooth
IF acc_std is Low AND acc_rms is Medium AND z_peak_to_peak is High AND speed is High THEN road = smooth
IF acc_std is Low AND acc_rms is High AND z_peak_to_peak is Low AND speed is High THEN road = smooth
IF acc_std is Low AND acc_rms is High AND z_peak_to_peak is Medium AND spe

In [79]:
#storing them 

In [80]:
rules = []

certain_rules = decision_table[decision_table["conflict"] == False]

for _, row in certain_rules.iterrows():
    
    rule = {
        "acc_std_bin": str(row["acc_std_bin"]),
        "acc_rms_bin": str(row["acc_rms_bin"]),
        "z_peak_to_peak_bin": str(row["z_peak_to_peak_bin"]),
        "speed_mean_bin": str(row["speed_mean_bin"]),
        "decision": list(set(row["road_label"]))[0]
    }
    
    rules.append(rule)

print("Total deterministic rules:", len(rules))

Total deterministic rules: 26


In [81]:
import pandas as pd

rules_df = pd.DataFrame(rules)

rules_df.to_csv("D:\\Coding\\6th sem\\Microprocessor\\road_quality\\rules\\ruleshybrid_rst_rules.csv", index=False)

print("Rules saved as CSV.")

Rules saved as CSV.


In [82]:
import json

with open("D:\\Coding\\6th sem\\Microprocessor\\road_quality\\rules\\hybrid_rst_rules.json", "w") as f:
    json.dump(rules, f, indent=4)

print("Rules saved as JSON.")

Rules saved as JSON.


In [83]:
rule = {
    "acc_std_bin": str(row["acc_std_bin"]),
    "acc_rms_bin": str(row["acc_rms_bin"]),
    "z_peak_to_peak_bin": str(row["z_peak_to_peak_bin"]),
    "speed_mean_bin": str(row["speed_mean_bin"]),
    "decision": list(set(row["road_label"]))[0],
    "support": len(row["road_label"])
}